<a href="https://colab.research.google.com/github/S00278393/secondrepo/blob/main/3_Streaming_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 3: ECG Streaming Pipeline with Spark Streaming and Kafka

This notebook implements a real-time ECG data streaming pipeline using:
- **Apache Kafka** as the message broker
- **Spark Structured Streaming** for real-time data processing
- **SQLite** for metrics persistence
- **Grafana** for real-time dashboard visualization

**Pipeline Architecture:**
```
ECG Data Generator -> Kafka Producer -> Kafka Topic ("ecg-stream")
    -> Spark Structured Streaming Consumer -> Windowed Aggregation
    -> SQLite Database -> Grafana Dashboard
```

**Note:** This notebook includes a demo mode that simulates the streaming
pipeline without requiring a running Kafka broker, making it runnable
in Google Colab.

## 1. Environment Setup

In [ ]:
# Install Java and Apache Spark
!apt-get update -qq
!apt-get install openjdk-11-jdk-headless -qq > /dev/null 2>&1
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar -xzf spark-3.5.1-bin-hadoop3.tgz
!pip install -q pyspark==3.5.1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 12.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


In [ ]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"
os.environ["PATH"] += ":/content/spark-3.5.1-bin-hadoop3/bin"

In [ ]:
# Install Kafka client library
!pip install confluent-kafka -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 40.2 MB/s eta 0:00:00


## 2. Kafka Producer — Streaming ECG Records

In [ ]:
import json
import time
import numpy as np
from confluent_kafka import Producer

# --- CONFIGURATION ---
KAFKA_CONFIG = {
    "bootstrap_servers": "localhost:9092",
    "topic": "ecg-stream",
    "rate": 5,           # Records per second
    "num_records": 1000,
    "max_send": 50,      # Max records to stream in demo
    "demo_mode": True    # Set to False if a Kafka broker is running
}

def generate_streaming_ecg_records(num_records=1000, seed=42):
    """Generate synthetic ECG records for streaming."""
    np.random.seed(seed)
    diagnostic_classes = ["NORM", "MI", "STTC", "CD", "HYP"]
    records = []
    for i in range(num_records):
        hr = float(np.random.normal(75, 15))
        records.append({
            "ecg_id": i + 1,
            "patient_id": int(np.random.randint(1, 2000)),
            "age": round(float(np.random.normal(55, 15)), 1),
            "sex": int(np.random.choice([0, 1])),
            "diagnostic_class": str(np.random.choice(diagnostic_classes)),
            "heart_rate": round(hr, 2),
            "signal_mean": round(float(np.random.normal(0, 0.1)), 4),
            "signal_std": round(float(np.random.uniform(0.1, 0.5)), 4),
            "qrs_duration": round(float(np.random.normal(100, 20)), 2),
        })
    return records

def delivery_report(err, msg):
    """Kafka delivery callback."""
    if err is not None:
        print(f"Message delivery failed: {err}")

def stream_ecg_records(records, config):
    """Stream ECG records to Kafka topic or console (demo mode)."""
    if config["demo_mode"]:
        print("\n[DEMO MODE] Printing records to console...\n")
    else:
        producer = Producer({"bootstrap.servers": config["bootstrap_servers"]})

    interval = 1.0 / config["rate"]
    total = min(config["max_send"], len(records))

    print(f"Streaming {total} records at {config['rate']} rec/sec...")
    for i in range(total):
        record = records[i].copy()
        record["stream_timestamp"] = time.strftime("%Y-%m-%dT%H:%M:%S")
        record["sequence_number"] = i
        message = json.dumps(record)

        if config["demo_mode"]:
            print(f"  [{i+1}/{total}] {message[:100]}...")
        else:
            producer.produce(
                config["topic"], key=str(record["ecg_id"]),
                value=message, callback=delivery_report
            )
            producer.poll(0)
        time.sleep(interval)

    if not config["demo_mode"]:
        producer.flush()
    print(f"\nFinished streaming {total} records.")

# Run the producer
print("=" * 60)
print("ECG Kafka Producer")
print("=" * 60)
data = generate_streaming_ecg_records(num_records=KAFKA_CONFIG["num_records"])
stream_ecg_records(data, KAFKA_CONFIG)

ECG Kafka Producer

[DEMO MODE] Printing records to console...

Streaming 50 records at 5 rec/sec...
  [1/50] {"ecg_id": 1, "patient_id": 1131, "age": 52.9, "sex": 1, "diagnostic_class": "HYP", "heart_rate": 82...
  [2/50] {"ecg_id": 2, "patient_id": 872, "age": 47.6, "sex": 1, "diagnostic_class": "STTC", "heart_rate": 44...
  [3/50] {"ecg_id": 3, "patient_id": 22, "age": 41.1, "sex": 0, "diagnostic_class": "CD", "heart_rate": 66.43...
  [4/50] {"ecg_id": 4, "patient_id": 958, "age": 52.5, "sex": 0, "diagnostic_class": "CD", "heart_rate": 71.2...
  [5/50] {"ecg_id": 5, "patient_id": 167, "age": 60.6, "sex": 1, "diagnostic_class": "CD", "heart_rate": 57.7...
  [6/50] {"ecg_id": 6, "patient_id": 1452, "age": 38.5, "sex": 0, "diagnostic_class": "STTC", "heart_rate": 8...
  [7/50] {"ecg_id": 7, "patient_id": 1130, "age": 25.6, "sex": 1, "diagnostic_class": "HYP", "heart_rate": 78...
  [8/50] {"ecg_id": 8, "patient_id": 338, "age": 31.3, "sex": 0, "diagnostic_class": "HYP", "heart_rate": 59

## 3. Spark Structured Streaming Consumer

In [ ]:
import json
import time
import datetime
import sqlite3
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# --- CONFIGURATION ---
STREAM_CONFIG = {
    "bootstrap_servers": "localhost:9092",
    "topic": "ecg-stream",
    "demo_mode": True  # Set to False with a running Kafka broker
}

DB_PATH = "/content/ecg_metrics.db"

# --- SPARK SESSION ---
def create_streaming_spark_session():
    return (
        SparkSession.builder
        .appName("ECG_Streaming_Consumer")
        .master("local[*]")
        .config("spark.jars.packages",
                "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
        .getOrCreate()
    )

def get_streaming_schema():
    """Schema for incoming ECG stream events."""
    return StructType([
        StructField("ecg_id", IntegerType(), False),
        StructField("patient_id", IntegerType(), False),
        StructField("age", DoubleType(), True),
        StructField("sex", IntegerType(), True),
        StructField("diagnostic_class", StringType(), True),
        StructField("heart_rate", DoubleType(), True),
        StructField("signal_mean", DoubleType(), True),
        StructField("signal_std", DoubleType(), True),
        StructField("qrs_duration", DoubleType(), True),
        StructField("stream_timestamp", StringType(), True),
        StructField("sequence_number", IntegerType(), True),
    ])

# --- ANALYTICS ---
def apply_anomaly_detection(df):
    """Flag anomalous ECG readings based on clinical thresholds."""
    return (
        df
        .withColumn("hr_anomaly",
            (F.col("heart_rate") < 50) | (F.col("heart_rate") > 150))
        .withColumn("qrs_anomaly", F.col("qrs_duration") > 140)
        .withColumn("is_anomalous",
            F.col("hr_anomaly") | F.col("qrs_anomaly"))
    )

def apply_windowed_agg(df):
    """
    Apply time-window aggregation following the Dataflow Model
    (Akidau et al., 2015) for handling unbounded, out-of-order data.

    Uses sliding windows with watermarking for late data handling.
    """
    df_ts = df.withColumn(
        "event_time",
        F.to_timestamp("stream_timestamp", "yyyy-MM-dd'T'HH:mm:ss")
    )
    return (
        df_ts
        .withWatermark("event_time", "30 seconds")
        .groupBy(
            F.window("event_time", "1 minute", "30 seconds"),
            "diagnostic_class"
        )
        .agg(
            F.count("*").alias("record_count"),
            F.round(F.avg("heart_rate"), 2).alias("avg_hr"),
            F.max("heart_rate").alias("peak_hr")
        )
    )

def save_to_sqlite(df, epoch_id):
    """Write Spark micro-batches to SQLite for Grafana polling."""
    pandas_df = df.toPandas()
    if "window" in pandas_df.columns:
        pandas_df["window_start"] = pandas_df["window"].apply(lambda x: x["start"])
        pandas_df.drop(columns=["window"], inplace=True)
    conn = sqlite3.connect(DB_PATH)
    pandas_df.to_sql("windowed_stats", conn, if_exists="append", index=False)
    conn.close()

In [ ]:
# --- RUN STREAMING PIPELINE ---
spark = create_streaming_spark_session()
schema = get_streaming_schema()

if STREAM_CONFIG["demo_mode"]:
    print("=" * 60)
    print("[DEMO MODE] Running batch simulation of streaming pipeline")
    print("=" * 60)

    # Generate test stream data
    test_data = []
    current_time = datetime.datetime.now()
    for i in range(100):
        ts = (current_time + datetime.timedelta(seconds=i * 10)).strftime(
            "%Y-%m-%dT%H:%M:%S"
        )
        row = {
            "ecg_id": int(i + 1),
            "patient_id": int(1000 + i),
            "age": float(np.random.uniform(20, 85)),
            "sex": int(np.random.choice([0, 1])),
            "diagnostic_class": str(
                np.random.choice(["NORM", "MI", "STTC", "CD", "HYP"])
            ),
            "heart_rate": float(np.random.uniform(40, 165)),
            "signal_mean": 0.0,
            "signal_std": 0.25,
            "qrs_duration": float(np.random.uniform(80, 155)),
            "stream_timestamp": str(ts),
            "sequence_number": int(i),
        }
        test_data.append(row)

    pdf = pd.DataFrame(test_data)
    pdf = pdf[[f.name for f in schema.fields]]
    parsed_stream = spark.createDataFrame(pdf, schema=schema)

    # Anomaly detection
    print("\n--- CRITICAL ANOMALIES ---")
    anomalies = apply_anomaly_detection(parsed_stream).filter("is_anomalous == True")
    anomalies.select(
        "ecg_id", "diagnostic_class", "heart_rate", "qrs_duration"
    ).show(10)

    # Windowed aggregations
    print("\n--- WINDOWED AGGREGATIONS (Dataflow Model) ---")
    windowed = apply_windowed_agg(parsed_stream)
    windowed.orderBy("window").show(10, truncate=False)

    # Save to SQLite for Grafana
    save_to_sqlite(windowed, 0)
    print(f"\nMetrics saved to {DB_PATH}")

else:
    print("=" * 60)
    print("[LIVE MODE] Connecting to Kafka stream")
    print("=" * 60)

    raw_stream = (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", STREAM_CONFIG["bootstrap_servers"])
        .option("subscribe", STREAM_CONFIG["topic"])
        .option("startingOffsets", "latest")
        .load()
    )

    parsed_stream = (
        raw_stream
        .select(
            F.from_json(F.col("value").cast("string"), schema).alias("data")
        )
        .select("data.*")
    )

    processed_stream = apply_windowed_agg(parsed_stream)

    query = (
        processed_stream.writeStream
        .foreachBatch(save_to_sqlite)
        .outputMode("update")
        .start()
    )
    query.awaitTermination()

[DEMO MODE] Running batch simulation of streaming pipeline

--- CRITICAL ANOMALIES ---
+------+----------------+------------------+------------------+
|ecg_id|diagnostic_class|        heart_rate|      qrs_duration|
+------+----------------+------------------+------------------+
|     1|            NORM|102.94625930044437|140.12319423735255|
|     3|             HYP|150.61584335562083|123.27653012679187|
|     8|            NORM| 94.53055604553967|152.89631339448542|
|    13|              CD|161.10205518542207|154.39196429897314|
|    15|            STTC|143.30670655687248| 148.5323752477849|
|    16|            STTC| 44.82394524226673| 145.9160859434639|
|    17|              CD|152.23726309678153| 89.63867513330874|
|    18|             HYP| 92.54192364943736|147.13250217996833|
|    19|             HYP| 60.20471172307398|147.93870424994793|
|    20|            NORM|155.21583368145767|136.81017880423272|
+------+----------------+------------------+------------------+
only showing top 

## 4. Grafana Setup for Real-Time Visualization

The following cells install and configure Grafana with an ngrok tunnel
for accessing the dashboard from a browser.

In [ ]:
# Install Grafana and ngrok
!wget -q https://dl.grafana.com/oss/release/grafana_10.2.3_amd64.deb
!sudo dpkg -i grafana_10.2.3_amd64.deb
!sudo apt-get install -f -y -qq
!pip install pyngrok -q

import os
if os.path.exists("/usr/sbin/grafana-server"):
    print("Grafana installed successfully")
else:
    print("Grafana installation failed - check logs above")

Selecting previously unselected package grafana.
(Reading database ... 118671 files and directories currently installed.)
Preparing to unpack grafana_10.2.3_amd64.deb ...
Unpacking grafana (10.2.3) ...
dpkg: dependency problems prevent configuration of grafana:
 grafana depends on musl; however:
  Package musl is not installed.

dpkg: error processing package grafana (--install):
 dependency problems - leaving unconfigured
Errors were encountered while processing:
 grafana
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package musl:amd64.
(Reading database ...

In [ ]:
import subprocess
import time

# Stop any existing Grafana process on port 3000
!fuser -k 3000/tcp 2>/dev/null || true

# Setup directories
!mkdir -p /var/lib/grafana /var/log/grafana /etc/grafana
!chmod -R 777 /var/lib/grafana /var/log/grafana

print("Starting Grafana server...")
process = subprocess.Popen(
    ["/usr/sbin/grafana-server",
     "--homepath", "/usr/share/grafana",
     "--config", "/etc/grafana/grafana.ini"],
    stdout=open("grafana.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)
!curl -s http://localhost:3000/api/health

Starting Grafana server...


In [ ]:
!curl -s http://localhost:3000/api/health

{
  "commit": "1e84fede543acc892d2a2515187e545eb047f237",
  "database": "ok",
  "version": "10.2.3"
}

In [ ]:
from pyngrok import ngrok

# Set your ngrok auth token (get one free at https://dashboard.ngrok.com)
# Replace with your own token:
NGROK_TOKEN = "1v5R8BfcmN9S4hU3UcWopU03W9F_7SEQET4PVyiJQrqzyrkbz"
ngrok.set_auth_token(NGROK_TOKEN)

# Create tunnel to access Grafana remotely
ngrok.kill()
try:
    public_url = ngrok.connect(3000)
    print(f"Access Grafana dashboard at: {public_url}")
    print("\nCredentials: admin / admin")
except Exception as e:
    print(f"Tunnel creation failed: {e}")
    print("You can still access Grafana at http://localhost:3000")

Access Grafana dashboard at: NgrokTunnel: "https://f53d-34-138-36-138.ngrok-free.app" -> "http://localhost:3000"

Credentials: admin / admin


In [ ]:
!pkill grafana-server || true

In [ ]:
# 1. Use the grafana-cli to install the community SQLite plugin
!grafana-cli plugins install frser-sqlite-datasource

# 2. Restart the Grafana process to load the new plugin
# First, kill the current process on port 3000
!fuser -k 3000/tcp || true

# 3. Relaunch the server (using the same 'homepath' command from before)
import subprocess
import time
print("Restarting Grafana with SQLite plugin...")

process = subprocess.Popen([
    "/usr/sbin/grafana-server",
    "--homepath", "/usr/share/grafana",
    "--config", "/etc/grafana/grafana.ini",
    "cfg:default.paths.logs=/var/log/grafana",
    "cfg:default.paths.data=/var/lib/grafana",
    "cfg:default.paths.plugins=/var/lib/grafana/plugins"
], stdout=open("grafana.log", 'a'), stderr=subprocess.STDOUT)

time.sleep(10)
print("✅ Grafana restarted. Refresh your Ngrok link browser tab!")

✔ Downloaded and extracted frser-sqlite-datasource v4.0.2 zip successfully to /var/lib/grafana/plugins/frser-sqlite-datasource

Please restart Grafana after installing or removing plugins. Refer to Grafana documentation for instructions if necessary.

3000/tcp:            15767
Restarting Grafana with SQLite plugin...


✅ Grafana restarted. Refresh your Ngrok link browser tab!


### Grafana Dashboard Configuration

1. **Access Grafana**: Use the ngrok URL printed above
2. **Login**: Username `admin`, Password `admin` (change on first login)
3. **Add SQLite Data Source**:
   - Go to Configuration > Data Sources > Add data source
   - Select **SQLite**
   - Database path: `/content/ecg_metrics.db`
   - Click **Save & Test**
4. **Create Dashboard**: Add panels with these SQL queries:

**Average Heart Rate Over Time:**
```sql
SELECT window_start as time, avg_hr, diagnostic_class
FROM windowed_stats
ORDER BY window_start
```

**Record Count by Diagnostic Class:**
```sql
SELECT diagnostic_class, SUM(record_count) as total
FROM windowed_stats
GROUP BY diagnostic_class
```

**Peak Heart Rate Monitoring:**
```sql
SELECT window_start as time, peak_hr, diagnostic_class
FROM windowed_stats
ORDER BY window_start
```

## 5. Verify Data in SQLite

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)
try:
    count = pd.read_sql("SELECT count(*) as total FROM windowed_stats", conn)
    print(f"Total records in database: {count['total'][0]}")

    print("\nSample data:")
    sample = pd.read_sql(
        "SELECT window_start, avg_hr, peak_hr, diagnostic_class, record_count "
        "FROM windowed_stats ORDER BY window_start DESC LIMIT 10",
        conn
    )
    print(sample.to_string(index=False))
except Exception as e:
    print(f"Database query error: {e}")
finally:
    conn.close()

Total records in database: 118

Sample data:
       window_start  avg_hr    peak_hr diagnostic_class  record_count
2026-04-20 18:05:00  103.57 103.567361             STTC             1
2026-04-20 18:05:00  115.64 115.637703               MI             1
2026-04-20 18:05:00  133.34 133.342913             NORM             1
2026-04-20 18:04:30  133.34 133.342913             NORM             1
2026-04-20 18:04:30  145.18 145.179970               CD             1
2026-04-20 18:04:30  129.60 143.566282               MI             2
2026-04-20 18:04:30  103.57 103.567361             STTC             1
2026-04-20 18:04:30   45.04  45.041270              HYP             1
2026-04-20 18:04:00  135.92 145.179970               CD             2
2026-04-20 18:04:00   81.17  81.172258             STTC             1


## Architecture and Data Flow

```
+-------------------+     +----------------+     +-------------------------+
| ECG Data          | --> | Kafka Producer | --> | Kafka Topic             |
| Generator         |     | (confluent-    |     | ("ecg-stream")          |
| (Synthetic Data)  |     |  kafka)        |     |                         |
+-------------------+     +----------------+     +-------------------------+
                                                           |
                                                           v
+-------------------+     +----------------+     +-------------------------+
| Grafana           | <-- | SQLite         | <-- | Spark Structured        |
| Dashboard         |     | Database       |     | Streaming Consumer      |
| (Visualization)   |     | (ecg_metrics)  |     | - Anomaly Detection     |
+-------------------+     +----------------+     | - Windowed Aggregation  |
                                                  | - Watermarking          |
                                                  +-------------------------+
```

**Scalability:** Kafka partitions allow horizontal scaling of producers
and consumers. Spark Streaming scales across cluster nodes.

**Fault Tolerance:** Kafka replication ensures message durability.
Spark checkpointing enables recovery from failures. SQLite provides
local persistence.

**Dataflow Model:** The windowed aggregation uses sliding windows with
watermarking (30 seconds allowed lateness), following principles from
Akidau et al. (2015) for handling out-of-order data in unbounded streams.